# 🫀 CardioTwin — Deep Learning Model Exploration Suite (Colab T4 GPU)
### Digital Twin Challenge 2026 (Happiest Health) • Phase 2 Deep Learning Tournament

This notebook trains and benchmarks **advanced Deep Learning architectures** directly on **raw 100 Hz photoplethysmogram (PPG) waveforms** (1,000 samples @ 100 Hz per window across 2,271 patients).

---
### ⚡ Hardware Setup
Go to **Runtime → Change runtime type → Select T4 GPU**.

---
### 🔬 4 Deep Learning Architectures Included:
1. **`Standard_1D_CNN`:** Our verified baseline (4 Conv1D blocks, 48.63% Macro-F1).
2. **`Inception_1D`:** Multi-scale parallel kernels (3, 7, 15, 31) capturing high-frequency flutter and low-frequency respiration simultaneously.
3. **`ResNet_1D`:** Deep residual blocks with skip connections preventing gradient degradation.
4. **`CRNN_BiLSTM`:** 1D-CNN spatial feature extractor followed by a Bidirectional LSTM modeling long-range 10-second heartbeat cadence.

+ **Optional Biosignal Data Augmentation:** Random circular time-shifting (+/- 100 ms) and amplitude scaling (0.85x–1.15x).

In [ ]:
# 1. Environment & GPU Verification
import torch
import torch.nn as nn
from torch.utils.data import TensorDataset, DataLoader

import numpy as np
import matplotlib.pyplot as plt
import json, time, os

from sklearn.model_selection import StratifiedGroupKFold
from sklearn.metrics import accuracy_score, f1_score, classification_report, confusion_matrix, ConfusionMatrixDisplay
from sklearn.utils.class_weight import compute_class_weight
from sklearn.preprocessing import LabelEncoder

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('PyTorch Version:', torch.__version__)
print('Compute Device: ', device)
if device.type == 'cuda':
    print('GPU Model:      ', torch.cuda.get_device_name(0))
else:
    print('⚠️ Running on CPU! Select T4 GPU in Runtime -> Change runtime type for fast GPU training.')

## 2. Load Dataset (`unified_multimodal_dataset.npz`)
Drag and drop `unified_multimodal_dataset.npz` (16.6 MB) into the left Files sidebar in Colab.

In [ ]:
data_filename = 'unified_multimodal_dataset.npz'
if not os.path.exists(data_filename):
    print('Please upload unified_multimodal_dataset.npz (16.6 MB)...')
    from google.colab import files
    uploaded = files.upload()
else:
    print(f'[OK] Found {data_filename} on disk ({os.path.getsize(data_filename)/1e6:.1f} MB)')

data = np.load(data_filename, allow_pickle=True)
X_waveforms = data['X_waveforms']  # (4683, 1000, 1)
y_raw = data['y']
patient_ids = data['patient_ids']

# Convert to PyTorch Conv1d format: (N, 1, 1000)
X_tensor = torch.tensor(X_waveforms, dtype=torch.float32).permute(0, 2, 1)

canonical_classes = ['AFib', 'Asystole', 'Bradycardia', 'Cardiac_Paced', 'Normal', 'Tachycardia', 'V_Flutter_Fib', 'V_Tachycardia']
le = LabelEncoder()
le.fit(canonical_classes)
y_encoded = le.transform(y_raw)
class_names = list(le.classes_)

print(f'\n[+] Total Windows:  {len(y_encoded):,}')
print(f'[+] Unique Patients: {len(np.unique(patient_ids)):,}')
print(f'[+] Tensor Shape:    {list(X_tensor.shape)}')
print('\nClass Breakdown:')
for idx, c in enumerate(class_names):
    cnt = np.sum(y_encoded == idx)
    print(f'  {c:<15}: {cnt:>5} windows ({cnt/len(y_encoded)*100:>4.1f}%)')

## 3. Deep Learning Model Architectures
Choose your architecture and toggle data augmentation below:

In [ ]:
#@title Deep Learning Tournament Configuration
EXECUTION_MODE = "Automated Tournament (Benchmark All Models)" #@param ["Automated Tournament (Benchmark All Models)", "Single Model Evaluation"]
SELECTED_MODEL = "Inception_1D" #@param ["Standard_1D_CNN", "Inception_1D", "ResNet_1D", "CRNN_BiLSTM"]
MODEL_ARCH = SELECTED_MODEL
USE_DATA_AUGMENTATION = True #@param {type:"boolean"}
EPOCHS = 18 #@param {type:"slider", min:10, max:30, step:2}
BATCH_SIZE = 64 #@param [32, 64, 128]
LEARNING_RATE = 0.001 #@param {type:"number"}

if EXECUTION_MODE == "Automated Tournament (Benchmark All Models)":
    ARCHITECTURES = ["Standard_1D_CNN", "Inception_1D", "ResNet_1D", "CRNN_BiLSTM"]
    print(f'[TOURNAMENT MODE] Will benchmark {len(ARCHITECTURES)} architectures sequentially on GPU:')
    for a in ARCHITECTURES:
        print(f'  - {a}')
else:
    ARCHITECTURES = [SELECTED_MODEL]
    print(f'[SINGLE MODEL MODE] Evaluating: {SELECTED_MODEL}')

print(f'\n[CONFIG] Data Augmentation: {USE_DATA_AUGMENTATION}')
print(f'[CONFIG] Epochs per fold:   {EPOCHS}')
print(f'[CONFIG] Batch Size:        {BATCH_SIZE}')
print(f'[CONFIG] Learning Rate:     {LEARNING_RATE}')


In [ ]:
# ------------------------------------------------------------------
# Architecture 1: Standard 1D-CNN (Verified Baseline)
# ------------------------------------------------------------------
class Standard1DCNN(nn.Module):
    def __init__(self, num_classes=8):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv1d(1, 32, kernel_size=15, stride=1, padding=7), nn.BatchNorm1d(32), nn.ReLU(), nn.MaxPool1d(2), nn.Dropout(0.2),
            nn.Conv1d(32, 64, kernel_size=11, stride=1, padding=5), nn.BatchNorm1d(64), nn.ReLU(), nn.MaxPool1d(2), nn.Dropout(0.2),
            nn.Conv1d(64, 128, kernel_size=7, stride=1, padding=3), nn.BatchNorm1d(128), nn.ReLU(), nn.MaxPool1d(2), nn.Dropout(0.3),
            nn.Conv1d(128, 128, kernel_size=5, stride=1, padding=2), nn.BatchNorm1d(128), nn.ReLU(), nn.AdaptiveAvgPool1d(1)
        )
        self.classifier = nn.Sequential(
            nn.Flatten(), nn.Linear(128, 64), nn.ReLU(), nn.Dropout(0.3), nn.Linear(64, num_classes)
        )
    def forward(self, x):
        return self.classifier(self.features(x))

# ------------------------------------------------------------------
# Architecture 2: Multi-Scale Inception-1D
# Parallel kernels (3, 7, 15, 31) capture multiple frequency bands
# ------------------------------------------------------------------
class InceptionBlock1D(nn.Module):
    def __init__(self, in_channels, out_channels):
        super().__init__()
        branch_ch = out_channels // 4
        self.branch1 = nn.Sequential(nn.Conv1d(in_channels, branch_ch, kernel_size=3, padding=1), nn.BatchNorm1d(branch_ch), nn.ReLU())
        self.branch2 = nn.Sequential(nn.Conv1d(in_channels, branch_ch, kernel_size=7, padding=3), nn.BatchNorm1d(branch_ch), nn.ReLU())
        self.branch3 = nn.Sequential(nn.Conv1d(in_channels, branch_ch, kernel_size=15, padding=7), nn.BatchNorm1d(branch_ch), nn.ReLU())
        self.branch4 = nn.Sequential(nn.Conv1d(in_channels, branch_ch, kernel_size=31, padding=15), nn.BatchNorm1d(branch_ch), nn.ReLU())
        self.shortcut = nn.Conv1d(in_channels, out_channels, kernel_size=1) if in_channels != out_channels else nn.Identity()
    def forward(self, x):
        out = torch.cat([self.branch1(x), self.branch2(x), self.branch3(x), self.branch4(x)], dim=1)
        return out + self.shortcut(x)

class Inception1DCNN(nn.Module):
    def __init__(self, num_classes=8):
        super().__init__()
        self.stem = nn.Sequential(nn.Conv1d(1, 32, kernel_size=7, padding=3), nn.BatchNorm1d(32), nn.ReLU(), nn.MaxPool1d(2))
        self.inc1 = InceptionBlock1D(32, 64)
        self.pool1 = nn.Sequential(nn.MaxPool1d(2), nn.Dropout(0.2))
        self.inc2 = InceptionBlock1D(64, 128)
        self.pool2 = nn.Sequential(nn.MaxPool1d(2), nn.Dropout(0.3))
        self.inc3 = InceptionBlock1D(128, 128)
        self.gap = nn.AdaptiveAvgPool1d(1)
        self.classifier = nn.Sequential(nn.Flatten(), nn.Linear(128, 64), nn.ReLU(), nn.Dropout(0.3), nn.Linear(64, num_classes))
    def forward(self, x):
        x = self.stem(x)
        x = self.pool1(self.inc1(x))
        x = self.pool2(self.inc2(x))
        x = self.gap(self.inc3(x))
        return self.classifier(x)

# ------------------------------------------------------------------
# Architecture 3: ResNet-1D (Deep Residual Skip Connections)
# ------------------------------------------------------------------
class ResidualBlock1D(nn.Module):
    def __init__(self, channels, kernel_size=7):
        super().__init__()
        self.conv1 = nn.Conv1d(channels, channels, kernel_size, padding=kernel_size//2)
        self.bn1 = nn.BatchNorm1d(channels)
        self.relu = nn.ReLU()
        self.conv2 = nn.Conv1d(channels, channels, kernel_size, padding=kernel_size//2)
        self.bn2 = nn.BatchNorm1d(channels)
    def forward(self, x):
        return self.relu(x + self.bn2(self.conv2(self.relu(self.bn1(self.conv1(x))))))

class ResNet1D(nn.Module):
    def __init__(self, num_classes=8):
        super().__init__()
        self.in_block = nn.Sequential(nn.Conv1d(1, 32, kernel_size=15, padding=7), nn.BatchNorm1d(32), nn.ReLU(), nn.MaxPool1d(2))
        self.res1 = ResidualBlock1D(32)
        self.trans1 = nn.Sequential(nn.Conv1d(32, 64, kernel_size=1), nn.MaxPool1d(2), nn.Dropout(0.2))
        self.res2 = ResidualBlock1D(64)
        self.trans2 = nn.Sequential(nn.Conv1d(64, 128, kernel_size=1), nn.MaxPool1d(2), nn.Dropout(0.3))
        self.res3 = ResidualBlock1D(128)
        self.gap = nn.AdaptiveAvgPool1d(1)
        self.fc = nn.Sequential(nn.Flatten(), nn.Linear(128, 64), nn.ReLU(), nn.Dropout(0.3), nn.Linear(64, num_classes))
    def forward(self, x):
        x = self.in_block(x)
        x = self.trans1(self.res1(x))
        x = self.trans2(self.res2(x))
        x = self.gap(self.res3(x))
        return self.fc(x)

# ------------------------------------------------------------------
# Architecture 4: CRNN (Conv1D + Bidirectional LSTM)
# ------------------------------------------------------------------
class CRNN1D(nn.Module):
    def __init__(self, num_classes=8):
        super().__init__()
        self.conv = nn.Sequential(
            nn.Conv1d(1, 32, kernel_size=11, padding=5), nn.BatchNorm1d(32), nn.ReLU(), nn.MaxPool1d(4), nn.Dropout(0.2),
            nn.Conv1d(32, 64, kernel_size=7, padding=3), nn.BatchNorm1d(64), nn.ReLU(), nn.MaxPool1d(4), nn.Dropout(0.2)
        )  # 1000 -> 250 -> 62 time steps
        self.lstm = nn.LSTM(input_size=64, hidden_size=64, num_layers=2, batch_first=True, bidirectional=True, dropout=0.2)
        self.classifier = nn.Sequential(nn.Linear(128, 64), nn.ReLU(), nn.Dropout(0.3), nn.Linear(64, num_classes))
    def forward(self, x):
        x = self.conv(x)  # (B, 64, 62)
        x = x.permute(0, 2, 1)  # (B, 62, 64) for LSTM
        lstm_out, _ = self.lstm(x)  # (B, 62, 128)
        x_pool = torch.mean(lstm_out, dim=1)  # Temporal average pooling
        return self.classifier(x_pool)

# Factory Helper
def create_model(arch_name, num_classes=8):
    if arch_name == 'Standard_1D_CNN': return Standard1DCNN(num_classes)
    elif arch_name == 'Inception_1D':   return Inception1DCNN(num_classes)
    elif arch_name == 'ResNet_1D':      return ResNet1D(num_classes)
    elif arch_name == 'CRNN_BiLSTM':    return CRNN1D(num_classes)
    else: raise ValueError(f'Unknown model architecture: {arch_name}')


# Verify all architectures instantiate cleanly
print('Verifying model architectures on device:')
for arch in ARCHITECTURES:
    m = create_model(arch).to(device)
    params = sum(p.numel() for p in m.parameters())
    print(f'  [+] {arch:<18}: {params:,} parameters')


## 4. Run 5-Fold Patient-Isolated Cross-Validation on GPU

In [ ]:
# Biosignal Waveform Augmentation (Batch-level on GPU)
def augment_waveforms(batch_x):
    # 1. Circular time-shift jitter (+/- 50 to 150 ms)
    shift = np.random.randint(-15, 15)
    batch_x = torch.roll(batch_x, shifts=shift, dims=-1)
    # 2. Random amplitude scale (0.85x to 1.15x)
    scale = torch.empty(batch_x.size(0), 1, 1, device=batch_x.device).uniform_(0.85, 1.15)
    batch_x = batch_x * scale
    return batch_x

def train_epoch(model, dataloader, criterion, optimizer, apply_aug=False):
    model.train()
    running_loss, correct, total = 0.0, 0, 0
    for X_batch, y_batch in dataloader:
        X_batch, y_batch = X_batch.to(device), y_batch.to(device)
        if apply_aug:
            X_batch = augment_waveforms(X_batch)
        optimizer.zero_grad()
        logits = model(X_batch)
        loss = criterion(logits, y_batch)
        loss.backward()
        optimizer.step()
        running_loss += loss.item() * X_batch.size(0)
        preds = torch.argmax(logits, dim=1)
        correct += (preds == y_batch).sum().item()
        total += y_batch.size(0)
    return running_loss / total, correct / total

def evaluate(model, dataloader, criterion):
    model.eval()
    running_loss, all_preds, all_probs, all_targets = 0.0, [], [], []
    with torch.no_grad():
        for X_batch, y_batch in dataloader:
            X_batch, y_batch = X_batch.to(device), y_batch.to(device)
            logits = model(X_batch)
            loss = criterion(logits, y_batch)
            running_loss += loss.item() * X_batch.size(0)
            probs = torch.softmax(logits, dim=1).cpu().numpy()
            all_preds.extend(np.argmax(probs, axis=1))
            all_probs.extend(probs)
            all_targets.extend(y_batch.cpu().numpy())
    return running_loss / len(all_targets), accuracy_score(all_targets, all_preds), f1_score(all_targets, all_preds, average='macro', zero_division=0), np.array(all_preds), np.array(all_probs)

sgkf = StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=42)
tournament_results = {}

print('=' * 80)
print(f'STARTING 5-FOLD DEEP LEARNING TOURNAMENT ({len(ARCHITECTURES)} Architectures)')
print('=' * 80)
total_tournament_start = time.time()

for arch_idx, arch_name in enumerate(ARCHITECTURES, 1):
    print(f'\n>>> [{arch_idx}/{len(ARCHITECTURES)}] Evaluating Architecture: {arch_name} ...')
    oof_preds = np.zeros(len(y_encoded), dtype=int)
    oof_probs = np.zeros((len(y_encoded), len(class_names)), dtype=float)
    fold_f1_scores = []
    t_arch_start = time.time()

    for fold, (train_idx, val_idx) in enumerate(sgkf.split(X_tensor, y_encoded, groups=patient_ids)):
        y_train = y_encoded[train_idx]
        cw = compute_class_weight('balanced', classes=np.arange(len(class_names)), y=y_train)
        cw_tensor = torch.tensor(cw, dtype=torch.float32).to(device)

        train_loader = DataLoader(TensorDataset(X_tensor[train_idx], torch.tensor(y_train, dtype=torch.long)), batch_size=BATCH_SIZE, shuffle=True)
        val_loader = DataLoader(TensorDataset(X_tensor[val_idx], torch.tensor(y_encoded[val_idx], dtype=torch.long)), batch_size=BATCH_SIZE, shuffle=False)

        model = create_model(arch_name).to(device)
        criterion = nn.CrossEntropyLoss(weight=cw_tensor)
        optimizer = torch.optim.AdamW(model.parameters(), lr=LEARNING_RATE, weight_decay=1e-4)
        scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS, eta_min=1e-5)

        best_f1, best_preds, best_probs = 0.0, None, None
        for epoch in range(1, EPOCHS + 1):
            train_loss, train_acc = train_epoch(model, train_loader, criterion, optimizer, apply_aug=USE_DATA_AUGMENTATION)
            val_loss, val_acc, val_f1, preds, probs = evaluate(model, val_loader, criterion)
            scheduler.step()
            if val_f1 > best_f1:
                best_f1, best_preds, best_probs = val_f1, preds, probs

        oof_preds[val_idx] = best_preds
        oof_probs[val_idx] = best_probs
        fold_f1_scores.append(best_f1)
        print(f'    Fold {fold+1}/5 Best Macro-F1: {best_f1:.4f}')

    elapsed = time.time() - t_arch_start
    acc = accuracy_score(y_encoded, oof_preds)
    macro_f1 = f1_score(y_encoded, oof_preds, average='macro', zero_division=0)
    weighted_f1 = f1_score(y_encoded, oof_preds, average='weighted', zero_division=0)
    clf_dict = classification_report(y_encoded, oof_preds, target_names=class_names, output_dict=True, zero_division=0)

    tournament_results[arch_name] = {
        'architecture': arch_name,
        'accuracy': float(acc),
        'macro_f1': float(macro_f1),
        'weighted_f1': float(weighted_f1),
        'fold_f1_scores': [float(f) for f in fold_f1_scores],
        'oof_preds': oof_preds,
        'oof_probs': oof_probs,
        'classification_report': clf_dict,
        'time_seconds': float(elapsed)
    }
    print(f'  [DONE] {arch_name} -> Acc: {acc*100:.2f}%, Macro-F1: {macro_f1*100:.2f}% (Finished in {elapsed:.1f}s)')

print('\n' + '=' * 80)
print(f'All {len(ARCHITECTURES)} Architectures Completed in {time.time()-total_tournament_start:.1f} seconds!')
print('=' * 80)


## 5. Evaluation Benchmark & Leaderboard Comparison

In [ ]:
# Leaderboard Display & Ranking
sorted_archs = sorted(tournament_results.keys(), key=lambda k: tournament_results[k]['macro_f1'], reverse=True)
best_arch = sorted_archs[0]

print('=' * 95)
print('             CARDIOTWIN PHASE 2 DEEP LEARNING TOURNAMENT LEADERBOARD')
print('          (Evaluated on 4,683 10-second Windows across 2,271 Unique Patients)')
print('=' * 95)
header = f"{'Rank':<5} {'Architecture':<18} {'Accuracy':<10} {'Macro-F1':<11} {'VFib F1':<10} {'Asystole F1':<13} {'AFib F1':<10} {'Time':<8} {'Status'}"
print(header)
print('-' * 95)

for rank, arch in enumerate(sorted_archs, 1):
    res = tournament_results[arch]
    rep = res['classification_report']
    acc_str = f"{res['accuracy']*100:.2f}%"
    f1_str = f"{res['macro_f1']*100:.2f}%"
    vfib_str = f"{rep.get('V_Flutter_Fib', {}).get('f1-score', 0.0):.4f}"
    asyst_str = f"{rep.get('Asystole', {}).get('f1-score', 0.0):.4f}"
    afib_str = f"{rep.get('AFib', {}).get('f1-score', 0.0):.4f}"
    time_str = f"{res['time_seconds']:.1f}s"
    status = '[CHAMPION]' if rank == 1 else ''
    print(f"{rank:<5} {arch:<18} {acc_str:<10} {f1_str:<11} {vfib_str:<10} {asyst_str:<13} {afib_str:<10} {time_str:<8} {status}")

print('=' * 95)
print(f'\n[+] TOURNAMENT WINNER: [{best_arch}] with {tournament_results[best_arch]["macro_f1"]*100:.2f}% Macro-F1!')

# Confusion Matrix for the Champion Model
champ_preds = tournament_results[best_arch]['oof_preds']
fig, ax = plt.subplots(figsize=(10, 8), dpi=200)
disp = ConfusionMatrixDisplay.from_predictions(
    y_encoded, champ_preds, display_labels=class_names,
    cmap=plt.cm.Greens, normalize='true', ax=ax, values_format='.2f'
)
plt.title(f'CardioTwin Deep Learning Champion: [{best_arch}]\n(5-Fold Patient-Isolated CV on 2,271 Patients - Macro-F1: {tournament_results[best_arch]["macro_f1"]*100:.2f}%)', fontsize=11, fontweight='bold', pad=15)
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.savefig('confusion_matrix_cnn.png')
plt.show()


## 6. Train Final Production Model on Full Dataset & Auto-Download Artifacts

In [ ]:
print(f'[+] Training Final Production Champion [{best_arch}] on full dataset (2,271 patients)...')
full_cw = compute_class_weight('balanced', classes=np.arange(len(class_names)), y=y_encoded)
full_loader = DataLoader(TensorDataset(X_tensor, torch.tensor(y_encoded, dtype=torch.long)), batch_size=BATCH_SIZE, shuffle=True)

final_model = create_model(best_arch).to(device)
final_criterion = nn.CrossEntropyLoss(weight=torch.tensor(full_cw, dtype=torch.float32).to(device))
final_optimizer = torch.optim.AdamW(final_model.parameters(), lr=LEARNING_RATE, weight_decay=1e-4)
final_scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(final_optimizer, T_max=EPOCHS, eta_min=1e-5)

for epoch in range(1, EPOCHS + 1):
    train_epoch(final_model, full_loader, final_criterion, final_optimizer, apply_aug=USE_DATA_AUGMENTATION)
    final_scheduler.step()

# 1. Save PyTorch weights (.pt)
torch.save(final_model.state_dict(), 'ppg_1d_cnn.pt')
print(f'[+] Saved champion weights to ppg_1d_cnn.pt ({sum(p.numel() for p in final_model.parameters()):,} parameters)')

# 2. Save metadata JSON with full tournament summary
champ_data = tournament_results[best_arch]
meta = {
    'model_name': f'CardioTwin_{best_arch}',
    'architecture': best_arch,
    'augmented': USE_DATA_AUGMENTATION,
    'version': '2.0.0-deep-champion',
    'input_shape': [1, 1000],
    'total_parameters': sum(p.numel() for p in final_model.parameters()),
    'validation_scheme': '5-Fold StratifiedGroupKFold on Patient IDs (Zero Patient-Leakage)',
    'total_patients': int(len(np.unique(patient_ids))),
    'total_windows': int(len(y_encoded)),
    'classes': class_names,
    'metrics': {
        'overall_accuracy': float(champ_data['accuracy']),
        'macro_f1': float(champ_data['macro_f1']),
        'weighted_f1': float(champ_data['weighted_f1']),
        'fold_f1_scores': champ_data['fold_f1_scores']
    },
    'per_class_metrics': champ_data['classification_report'],
    'tournament_leaderboard': [
        {
            'rank': r,
            'architecture': a,
            'macro_f1': float(tournament_results[a]['macro_f1']),
            'accuracy': float(tournament_results[a]['accuracy']),
            'time_seconds': float(tournament_results[a]['time_seconds'])
        }
        for r, a in enumerate(sorted_archs, 1)
    ]
}
with open('cnn_metadata.json', 'w') as f:
    json.dump(meta, f, indent=2)
print('[+] Saved tournament scorecard to cnn_metadata.json')

# 3. Save Out-Of-Fold (OOF) predictions for leak-free dual-modality fusion
oof_file = 'oof_predictions_cnn.npz'
np.savez_compressed(
    oof_file,
    oof_probs=champ_data['oof_probs'],
    oof_preds=champ_data['oof_preds'],
    y_true=y_encoded,
    patient_ids=patient_ids,
    class_names=np.array(class_names),
    champion_name=best_arch
)
print(f'[+] Saved leak-free Out-Of-Fold (OOF) predictions to {oof_file} (Shape: {champ_data["oof_probs"].shape})')

# 3. Trigger automatic download of artifacts
try:
    from google.colab import files
    print('\nDownloading champion artifacts to your computer...')
    files.download('ppg_1d_cnn.pt')
    files.download('cnn_metadata.json')
    files.download('confusion_matrix_cnn.png')
    files.download('oof_predictions_cnn.npz')
except Exception as e:
    print('Download manually from the left sidebar if auto-download does not trigger:', e)
